# Benchmarks in the suite

One row per benchmark enabled in `config/eval.yaml`: domain, what it tests, size per language, where each language comes from and how it was made, and the source link.

- **Samples** are counted from the built files in `data/` (run `uv run python prepare.py` first): `ready` = items with a prompt, `default` = what a run uses after the config's filter. Items still waiting for translation are not counted.
- **Provenance** is hand-written below (from the papers and dataset cards, see `reports/{name}.md`). Categories: `native` (written in that language), `human` (human translation), `LLM + human` (LLM draft, then edited or checked by people), `LLM` (LLM translation, no human check by the authors), `ours` (translated in this project), `templated` (produced by a generator from a reviewed template), `pending` (to translate).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yaml

# the kernel may start in notebooks/ or at the repo root
here = Path.cwd().resolve()
ROOT = next(d for d in [here, *here.parents] if (d / "config/eval.yaml").exists() and (d / "run_eval.py").exists())
sys.path.insert(0, str(ROOT))
from run_eval import DOMAINS, select  # noqa: E402

config = yaml.safe_load((ROOT / "config/eval.yaml").read_text(encoding="utf-8"))
enabled = {name: spec for name, spec in config["benchmarks"].items() if spec.get("enabled", True)}
pd.set_option("display.max_colwidth", None)
print(len(enabled), "benchmarks enabled:", ", ".join(enabled))

In [ ]:
HF = "https://huggingface.co/datasets/"
PROVENANCE = {
    "polymath": {"title": "PolyMath", "tests": "math word problems at four difficulty levels, school to olympiad; default: medium + high (competition level)",
                "it": "LLM + human", "en": "native", "links": [HF + "Qwen/PolyMath"],
                     "detail": "GPT-4o draft, edited by a professional translation team (Italian); arXiv:2504.18428 §3, App. A.1"},
    "mclm_math100": {"title": "MCLM MT-MATH100", "tests": "high-school competition math from MATH-500: algebra, geometry, number theory, counting, precalculus; levels 1-5",
                    "it": "LLM", "en": "native", "links": [HF + "amphora/MCLM", HF + "HuggingFaceH4/MATH-500"],
                         "detail": "GPT-4o translation of MATH-500 problems; 2 dropped figures restored by us"},
    "mclm_aime2024": {"title": "MCLM MT-AIME2024", "tests": "AIME 2024: hard olympiad-qualifier problems, integer answers 0-999; widely used, likely contaminated",
                     "it": "LLM", "en": "native", "links": [HF + "amphora/MCLM"],
                          "detail": "GPT-4o translation of AIME 2024; II-08 broken (off)"},
    "aime25": {"title": "AIME 2025", "tests": "AIME 2025: same kind as AIME 2024, more recent so less likely seen in training",
              "it": "LLM + human", "en": "native", "links": [HF + "MathArena/aime_2025", HF + "fedric95/AIME2025-Multilingual"],
                   "detail": "best of 4 LLM candidates (GPT-4o, Claude 3.7 Sonnet), then checked by hand by the authors"},
    "invalsi_mate": {"title": "INVALSI MATE", "tests": "Italian national school math tests, grades 2-13; multiple choice, true/false and numeric items, written natively in Italian",
                    "it": "native", "en": None, "links": [HF + "ai4text/Invalsi"],
                         "detail": "Italian national school tests; Italian only"},
    "gpqa": {"title": "GPQA main", "tests": "graduate-level biology, physics and chemistry questions written to resist web search; 4 options",
            "it": "LLM", "en": "native", "links": [HF + "Idavidrein/gpqa", HF + "ellamind/gpqa-multilingual"],
                 "detail": "Gemini 3.1 Pro translation (ellamind); distractors realigned by us; 39 items flagged by the translator; gated"},
    "humaneval": {"title": "HumanEval+", "tests": "write a short Python function from its docstring; hidden unit tests",
                 "it": "ours (LLM)", "en": "native", "links": [HF + "evalplus/humanevalplus"],
                      "detail": "our translation, TranslateGemma 27B on Cineca (docstring prose only), every row re-checked at build"},
    "mbppplus": {"title": "MBPP+", "tests": "basic Python programming tasks from a one-line description; hidden unit tests",
                "it": "ours (LLM)", "en": "native", "links": [HF + "evalplus/mbppplus"],
                     "detail": "our translation, TranslateGemma 27B on Cineca (task text only), every row re-checked at build"},
    "livecodebench": {"title": "LiveCodeBench (release_v3)", "tests": "competitive programming (LeetCode, AtCoder, Codeforces, 2023-05 to 2024-08), easy to hard; hidden tests",
                     "it": "ours (LLM)", "en": "native", "links": [HF + "livecodebench/code_generation_lite"],
                          "detail": "our translation, TranslateGemma 27B on Cineca; output words in Italian (Yes -> Sì), test outputs mapped; 5 sentences repaired by hand"},
    "multi_if": {"title": "Multi-IF (turn 1)", "tests": "following verifiable format constraints (word counts, keywords, no commas, JSON...); first turn only",
                "it": "LLM + human", "en": "native", "links": [HF + "facebook/Multi-IF"],
                     "detail": "Llama 3.1 405B translation, every item checked by annotators, ~15% rewritten (arXiv:2410.15553 §3.3); 5 items repaired by us"},
    "aiw": {"title": "Alice in Wonderland", "tests": "one short common-sense relational problem ('Alice has N brothers and M sisters...') in variations; robustness of basic reasoning",
           "it": "ours", "en": "native", "links": ["https://github.com/LAION-AI/AIW"],
                "detail": "our translation of the 30 prompts, native-read 2026-10-16"},
    "xalpaca": {"title": "X-AlpacaEval", "tests": "open-ended assistant requests; answers judged pairwise against a reference model's",
               "it": "human", "en": "native", "links": [HF + "zhihz0535/X-AlpacaEval"],
                    "detail": "professional human translation of the 805 AlpacaEval instructions; 1 item repaired by us"},
    "zebra_gen": {"title": "Zebra (templated)", "tests": "logic grid puzzles (constraint satisfaction), 2-6 houses x 2-6 attributes; size scales the search, the deduction stays the same",
                 "it": "templated", "en": "templated", "links": ["https://github.com/alexandrainst/zebra_puzzles"],
                      "detail": "same seed, same puzzle in both languages; Italian config native-reviewed 2026-10-16"},
}
missing = set(enabled) - set(PROVENANCE)
assert not missing, f"add provenance for {missing}"

In [ ]:
rows = []
for name, spec in enabled.items():
    p = PROVENANCE[name]
    ready = {lang: len(select(name, lang, {}, None)) for lang in ("it", "en")}
    default = {lang: len(select(name, lang, spec, None)) if lang in spec["langs"] else 0 for lang in ("it", "en")}
    rows.append({
        "domain": DOMAINS.get(name, "other"),
        "tests": p["tests"],
        "benchmark": p["title"],
        "name": name,
        "it (default / ready)": f"{default['it']} / {ready['it']}" if ready["it"] else "–",
        "en (default / ready)": f"{default['en']} / {ready['en']}" if ready["en"] else "–",
        "languages": ", ".join(lang for lang in ("it", "en") if ready[lang]) or "–",
        "it source": p["it"],
        "en source": p["en"] or "–",
        "how the Italian was made": p["detail"],
        "link": "  ".join(p["links"]),
    })
df = pd.DataFrame(rows).sort_values(["domain", "benchmark"]).reset_index(drop=True)
df.to_csv(ROOT / "notebooks" / "benchmarks.csv", index=False)
df

## Per domain

In [ ]:
def counts(name, lang):
    spec = enabled[name]
    return len(select(name, lang, spec, None)) if lang in spec["langs"] else 0

df["it n"] = [counts(n, "it") for n in df["name"]]
df["en n"] = [counts(n, "en") for n in df["name"]]
df.groupby("domain").agg(benchmarks=("name", "count"), it_items=("it n", "sum"), en_items=("en n", "sum"))

## Italian side by provenance

In [ ]:
df.groupby("it source").agg(benchmarks=("benchmark", ", ".join), it_items=("it n", "sum"))